# Projet COVID 19 : Modelisation Random Forest - 4 classes - Avec correction du déséquillibre des classes

## Environnement de travail

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier

from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    PrecisionRecallDisplay
)

from sklearn.inspection import permutation_importance
import joblib

In [9]:
# Répertoire du notebook
NOTEBOOK_DIR = Path.cwd()

# Racine du projet (Liora_Covid)
PROJECT_ROOT = NOTEBOOK_DIR.parent.parent

# Dossier des CSV
DATA_DIR = PROJECT_ROOT / "csv"

print("Notebook :", NOTEBOOK_DIR)
print("Projet   :", PROJECT_ROOT)
print("CSV      :", DATA_DIR)

Notebook : C:\Users\n_a_e\Documents\DataScientest\Data Scientist\Projet COVID\Liora_Covid\notebooks\Modelisation
Projet   : C:\Users\n_a_e\Documents\DataScientest\Data Scientist\Projet COVID\Liora_Covid
CSV      : C:\Users\n_a_e\Documents\DataScientest\Data Scientist\Projet COVID\Liora_Covid\csv


In [8]:
# Chargement des données
train = pd.read_csv(DATA_DIR / "train_features.csv")
validation = pd.read_csv(DATA_DIR / "validation_features.csv")
test = pd.read_csv(DATA_DIR / "test_features.csv")

print(f"\nTrain      : {train.shape}")
print(f"Validation : {validation.shape}")
print(f"Test       : {test.shape}")


Train      : (16668, 9)
Validation : (2083, 9)
Test       : (2084, 9)


## Définition de la cible et des features

In [17]:
target = "classe"

features = [
    "pixel_mean",
    "pixel_std",
    "lum_interieur_masque",
    "lum_exterieur_masque",
    "surface_masque",
    "variance_laplacien"
]

target = "classe"

X_train = train[features]
y_train = train[target]

X_val = validation[features]
y_val = validation[target]

X_test = test[features]
y_test = test[target]

## Vérification des classes

In [18]:
print("Distribution train :")
print(y_train.value_counts(normalize=True))

print("\nDistribution validation :")
print(y_val.value_counts(normalize=True))

print("\nDistribution test :")
print(y_test.value_counts(normalize=True))

Distribution train :
classe
Normal             0.486501
Lung_Opacity       0.287437
COVID              0.161867
Viral Pneumonia    0.064195
Name: proportion, dtype: float64

Distribution validation :
classe
Normal             0.486318
Lung_Opacity       0.287086
COVID              0.162266
Viral Pneumonia    0.064330
Name: proportion, dtype: float64

Distribution test :
classe
Normal             0.486564
Lung_Opacity       0.287428
COVID              0.161708
Viral Pneumonia    0.064299
Name: proportion, dtype: float64


## Pipeline Random Forest

In [13]:
rf_pipe = Pipeline([
    ("model", RandomForestClassifier(
        random_state=0,
        class_weight="balanced",
        n_jobs=-1
    ))
])

## Grille des Hyperparamètres

In [14]:
param_grid = {
    "model__n_estimators": [100, 200, 300],
    "model__max_depth": [None, 5, 10, 20],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 4],
    "model__max_features": ["sqrt", "log2"]
}

## Cross-validation stratifiée + GridSearch

In [26]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=0
)

grid_rf = GridSearchCV(
    estimator=rf_pipe,
    param_grid=param_grid,
    scoring="f1_macro",
    cv=cv,
    n_jobs=-1,
    verbose=1,
    return_train_score=True
)

grid_rf.fit(X_train, y_train)


Fitting 5 folds for each of 216 candidates, totalling 1080 fits


GridSearchCV(cv=StratifiedKFold(n_splits=5, random_state=0, shuffle=True),
             estimator=Pipeline(steps=[('model',
                                        RandomForestClassifier(class_weight='balanced',
                                                               n_jobs=-1,
                                                               random_state=0))]),
             n_jobs=-1,
             param_grid={'model__max_depth': [None, 5, 10, 20],
                         'model__max_features': ['sqrt', 'log2'],
                         'model__min_samples_leaf': [1, 2, 4],
                         'model__min_samples_split': [2, 5, 10],
                         'model__n_estimators': [100, 200, 300]},
             return_train_score=True, scoring='f1_macro', verbose=1)

In [28]:
# Meilleurs paramètres

print("Meilleurs paramètres :")
print(grid_rf.best_params_)

print("\nMeilleur score F1 moyen en cross-validation :")
print(grid_rf.best_score_)

Meilleurs paramètres :
{'model__max_depth': None, 'model__max_features': 'sqrt', 'model__min_samples_leaf': 4, 'model__min_samples_split': 2, 'model__n_estimators': 200}

Meilleur score F1 moyen en cross-validation :
0.49861727696884184


In [29]:
# Résultats de la GridSearch

results = pd.DataFrame(grid_rf.cv_results_)
results = results.sort_values("rank_test_score")

results[[
    "rank_test_score",
    "mean_test_score",
    "std_test_score",
    "mean_train_score",
    "std_train_score",
    "params"
]].head(10)

,rank_test_score,mean_test_score,std_test_score,mean_train_score,std_train_score,params
22,1,0.498617,0.010174,0.872720,0.001839,"{'model__max_depth': None, 'model__max_feature..."
19,1,0.498617,0.010174,0.872720,0.001839,"{'model__max_depth': None, 'model__max_feature..."
49,1,0.498617,0.010174,0.872720,0.001839,"{'model__max_depth': None, 'model__max_feature..."
46,1,0.498617,0.010174,0.872720,0.001839,"{'model__max_depth': None, 'model__max_feature..."
47,5,0.497475,0.008969,0.873659,0.002280,"{'model__max_depth': None, 'model__max_feature..."
50,5,0.497475,0.008969,0.873659,0.002280,"{'model__max_depth': None, 'model__max_feature..."
23,5,0.497475,0.008969,0.873659,0.002280,"{'model__max_depth': None, 'model__max_feature..."
20,5,0.497475,0.008969,0.873659,0.002280,"{'model__max_depth': None, 'model__max_feature..."
212,9,0.496982,0.008905,0.859390,0.002405,"{'model__max_depth': 20, 'model__max_features'..."
209,9,0.496982,0.008905,0.859390,0.002405,"{'model__max_depth': 20, 'model__max_features'..."


## Évaluation sur validation

In [30]:
best_rf = grid_rf.best_estimator_

y_val_pred = best_rf.predict(X_val)
y_val_proba = best_rf.predict_proba(X_val)

print("Évaluation sur le jeu de validation")
print("Accuracy :", accuracy_score(y_val, y_val_pred))
print("Balanced accuracy :", balanced_accuracy_score(y_val, y_val_pred))
print("Precision macro :", precision_score(y_val, y_val_pred, average="macro"))
print("Recall macro :", recall_score(y_val, y_val_pred, average="macro"))
print("F1-score macro :", f1_score(y_val, y_val_pred, average="macro"))
print("F1-score weighted :", f1_score(y_val, y_val_pred, average="weighted"))

print("\nRapport de classification :")
print(classification_report(y_val, y_val_pred))

Évaluation sur le jeu de validation
Accuracy : 0.586173787806049
Balanced accuracy : 0.4991780495889732
Precision macro : 0.48162171593920655
Recall macro : 0.4991780495889732
F1-score macro : 0.4843678861161387
F1-score weighted : 0.5710006447478796

Rapport de classification :
                 precision    recall  f1-score   support

          COVID       0.27      0.16      0.20       338
   Lung_Opacity       0.56      0.57      0.56       598
         Normal       0.68      0.75      0.71      1013
Viral Pneumonia       0.41      0.52      0.46       134

       accuracy                           0.59      2083
      macro avg       0.48      0.50      0.48      2083
   weighted avg       0.56      0.59      0.57      2083



In [31]:
# ROC AUC multiclasse
print("ROC AUC macro OvR :", roc_auc_score(
    y_val,
    y_val_proba,
    multi_class="ovr",
    average="macro"
))

ROC AUC macro OvR : 0.7667057229607512


## Métriques d'évaluation

In [32]:
# Matrice de confusion

class_names = best_rf.named_steps["model"].classes_

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=class_names
)
disp.plot(xticks_rotation=45)
plt.title("Matrice de confusion - Validation")
plt.show()

NameError: name 'cm' is not defined

In [20]:
# Courbe ROC

roc_auc_score(..., multi_class="ovr", average="macro")

In [ ]:
# Courbe Precision-Recall

PrecisionRecallDisplay.from_estimator(best_rf, X_val, y_val)
plt.title("Courbe Precision-Recall - Random Forest")
plt.show()

In [ ]:
# Features importance

rf_model = best_rf.named_steps["model"]

importances = rf_model.feature_importances_
feature_names = X_train.columns

feature_importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": importances
}).sort_values("importance", ascending=False)

feature_importance_df.head(20)

In [ ]:
plt.figure(figsize=(10, 8))
plt.barh(
    feature_importance_df["feature"].head(20)[::-1],
    feature_importance_df["importance"].head(20)[::-1]
)
plt.xlabel("Importance relative")
plt.title("Top 20 des variables les plus importantes - Random Forest")
plt.show()